# 1 · Prepare data — runs **here in the notebook** (no training job)

Builds the splits, shows samples, and uploads to S3. Key addition: the **irrelevance / no-tool-needed** examples are split DISJOINTLY —
- a held-out slice → `irrelevance.jsonl` (abstention metric at eval), and
- a separate slice (gold=`[]`) **mixed into `grpo_train.jsonl`** so GRPO learns *restraint* (not over-calling), which SFT on positive-only demos can't.

Reuses `../src/data_utils.py` + `../src/tool_reward.py`.

In [ ]:
# If you hit: AttributeError: module 'pyarrow' has no attribute 'PyExtensionType'
# the base image has a new pyarrow but an OLD `datasets`. Upgrade, then RESTART THE KERNEL:
# %pip install -U datasets pyarrow 'transformers>=4.57' huggingface_hub boto3

In [ ]:
import os
try:
    os.getcwd()
except (FileNotFoundError, OSError):
    os.chdir(os.path.expanduser('~')); print('cwd was invalid; switched to', os.getcwd())
import sys, json, random
sys.path.insert(0, '../src')
import data_utils, tool_reward as R, common

cfg = json.load(open('toolcall_config.json'))
N_EVAL_IRR = cfg.get('n_irrelevance', 200)
FRAC = cfg.get('grpo_irrelevance_frac', 0.25)
print('base model:', cfg['base_model'])
print('splits    :', dict(n_train=cfg['n_train'], n_grpo=cfg['n_grpo'], n_eval=cfg['n_eval']))
print('irrelevance: eval=%d, grpo_frac=%.2f' % (N_EVAL_IRR, FRAC))
print('upload to :', cfg['data_s3'])

from huggingface_hub import login
if cfg.get('hf_token'):
    login(token=cfg['hf_token'], add_to_git_credential=False); print('HF login OK')

### Load tokenizer + build the disjoint splits

In [ ]:
from transformers import AutoTokenizer
tokenizer = AutoTokenizer.from_pretrained(cfg['base_model'], trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

eval_rows, grpo_rows, sft_rows = data_utils.load_and_split(
    cfg['xlam_dataset'], tokenizer,
    n_eval=cfg['n_eval'], n_grpo=cfg['n_grpo'], n_train=cfg['n_train'],
    seed=cfg['seed'], enable_thinking=bool(cfg['enable_thinking']))
print(f'eval={len(eval_rows)}  grpo(tool-needed)={len(grpo_rows)}  sft={len(sft_rows)}')

### SFT samples

In [ ]:
def show_tail(p, n=600):
    print(p[-n:] if len(p) > n else p)
for i in range(2):
    ex = sft_rows[i]; print('=' * 90)
    print('SFT query:', ex['query']); print('COMPLETION:', ex['completion'])

### GRPO samples (tool-needed)

In [ ]:
for i in range(2):
    ex = grpo_rows[i]; print('=' * 90)
    print('GRPO query:', ex['query']); print('gold:', ex['gold'])

### Irrelevance / no-tool-needed — split DISJOINTLY (eval vs GRPO-train)
`build_irrelevance` pairs a real query with tools from a different row (none fit). Correct behavior = make **no** call (`gold='[]'`).

In [ ]:
n_grpo_irr = int(round(len(grpo_rows) * FRAC / (1 - FRAC))) if 0 < FRAC < 1 else 0
total_irr = data_utils.build_irrelevance(
    cfg['xlam_dataset'], tokenizer, N_EVAL_IRR + n_grpo_irr,
    cfg['seed'], bool(cfg['enable_thinking']))
eval_irr = total_irr[:N_EVAL_IRR]
grpo_irr = total_irr[N_EVAL_IRR:N_EVAL_IRR + n_grpo_irr]
print(f'irrelevance: {len(eval_irr)} held-out (eval) + {len(grpo_irr)} mixed into GRPO train')
for i in range(2):
    print('=' * 90)
    print('query (no offered tool fits):', eval_irr[i]['query'])
    print('PROMPT tail:'); print(eval_irr[i]['prompt'][-450:])
    print('expected: NO <tool_call>')

### Build the mixed GRPO training set + write everything

In [ ]:
grpo_mixed = [{'prompt': r['prompt'], 'gold': r['gold']} for r in grpo_rows] + \
             [{'prompt': r['prompt'], 'gold': '[]'} for r in grpo_irr]
random.Random(cfg['seed']).shuffle(grpo_mixed)
frac_irr = sum(1 for r in grpo_mixed if r['gold'] == '[]') / max(len(grpo_mixed), 1)
print(f'GRPO train: {len(grpo_mixed)} prompts, {frac_irr:.0%} irrelevance')

n_val = max(1, int(0.02 * len(sft_rows)))
sft_val, sft_train = sft_rows[:n_val], sft_rows[n_val:]
os.makedirs('data', exist_ok=True)
data_utils.write_jsonl('data/sft_train.jsonl', sft_train, ['prompt', 'completion'])
data_utils.write_jsonl('data/sft_val.jsonl',   sft_val,   ['prompt', 'completion'])
data_utils.write_jsonl('data/grpo_train.jsonl', grpo_mixed, ['prompt', 'gold'])
data_utils.write_jsonl('data/eval.jsonl',      eval_rows, ['prompt', 'gold', 'query'])
data_utils.write_jsonl('data/irrelevance.jsonl', eval_irr, ['prompt', 'gold', 'query'])
print('local files:', os.listdir('data'))

In [ ]:
common.s3_upload_dir('data', cfg['data_s3'])
print('\nuploaded ->', cfg['data_s3'])

**Next:** `02_sft.ipynb` → `03_grpo.ipynb` (now trains on tool-needed **+** irrelevance, so GRPO can recover abstention) → `04_evaluate.ipynb` (watch the `abstention` column: SFT crushes it, GRPO should bring it back while holding accuracy).